# 📖 Notebook 2: Load Balancing Algorithms Compared

In Notebook 1 we used plain **round robin**. It's simple and fair in terms of
*request count*, but it doesn't adapt when some requests are much more
expensive than others.

In this notebook we compare four classic algorithms on the same workload:

1. **Round Robin** — cycle through the list.
2. **Weighted Round Robin** — give stronger backends more turns.
3. **Least Connections** — send to whoever is currently least busy.
4. **Random** — pick any backend uniformly at random.

## Learning Objectives

By the end of this notebook, you'll understand:

- How each algorithm decides where to send a request
- Why "fair by request count" ≠ "fair by load"
- What **tail latency** means and why it matters more than the average
- When to pick which algorithm in the real world


## 🛠️ Setup

From the lab folder:

```bash
cd 01-foundations/load-balancing
uv sync
```

### Kernel selection

In VS Code, click the kernel picker at the **top-right** of this notebook and
choose the `.venv` interpreter (it will be named something like
`.venv (Python 3.x)`).

If the `.venv` kernel doesn't appear in the list, reload the VS Code window:

- `Cmd+Shift+P` (macOS) or `Ctrl+Shift+P` (Windows/Linux)
- Type and select **"Developer: Reload Window"**

No Docker or external services are needed for this lab — everything runs
in-process with plain Python.


## 🧱 Setting the stage

We'll reuse the idea of fake backends from Notebook 1, but this time each
backend has a **capacity** (how many "units" of work per second it can do).
Stronger backends finish the same work in less wall-clock time.

We'll also track **active_connections** — the number of requests currently
in-flight — so the least-connections algorithm has something to look at.


In [ ]:
from pydantic import BaseModel, Field
from typing import Literal
import random
import itertools
import heapq

class Request(BaseModel):
    """An incoming request with an amount of 'work' measured in work-units."""
    id: int
    work_units: float = Field(gt=0)


class Backend(BaseModel):
    """A pretend backend.

    `capacity` = how many work-units it can process per second. A capacity of
    2.0 means it finishes a 1-unit request in 0.5 seconds.
    """
    name: str
    capacity: float = Field(gt=0)
    active: int = 0        # currently in-flight requests
    total_latency: float = 0.0  # sum of response times (for stats)
    handled: int = 0

    def service_time(self, req: Request) -> float:
        """How long this backend would take to process `req`, alone."""
        return req.work_units / self.capacity


# A heterogeneous fleet — 2 beefy servers, 2 small ones
def fresh_backends() -> list[Backend]:
    return [
        Backend(name="big-1",   capacity=4.0),
        Backend(name="big-2",   capacity=4.0),
        Backend(name="small-1", capacity=1.0),
        Backend(name="small-2", capacity=1.0),
    ]


fresh_backends()


## 🎲 Building a realistic, uneven workload

Real traffic is **bursty** and **skewed**: most requests are small, but a few
are huge (think: a search query vs. generating a PDF report). We'll build a
workload that looks like that.


In [ ]:
random.seed(7)

def make_workload(n: int = 2000) -> list[Request]:
    """90% small requests, 10% 'heavy' requests (5-20x bigger)."""
    reqs = []
    for i in range(n):
        if random.random() < 0.10:
            work = random.uniform(5.0, 20.0)   # heavy
        else:
            work = random.uniform(0.1, 1.0)    # light
        reqs.append(Request(id=i, work_units=work))
    return reqs


workload = make_workload()
heavy = sum(1 for r in workload if r.work_units >= 5)
print(f"{len(workload)} requests, {heavy} of them are heavy ({heavy/len(workload):.0%})")
print(f"Total work: {sum(r.work_units for r in workload):.1f} units")


## ⏱️ A tiny event-driven simulator

To compare algorithms we need to know **when** each request finishes, not
just "who got it". We'll simulate time with a priority queue of events:

- Every 0.05 seconds a new request arrives.
- The load balancer picks a backend using its algorithm.
- The backend starts the request — it will finish at
  `now + work_units / capacity`.
- When a request finishes, the backend's `active` counter drops.

Latency for a request = `finish_time - arrival_time`. That's what users feel.


In [ ]:
def simulate(workload: list[Request], pick_backend) -> tuple[list[Backend], list[float]]:
    """Run `workload` through a load balancer that calls `pick_backend(backends, req)`.

    Returns:
      - the list of backends with stats filled in
      - a list of per-request latencies (seconds), in finish order
    """
    backends = fresh_backends()
    tiebreak = itertools.count()
    events: list[tuple[float, int, str, object]] = []

    ARRIVAL_INTERVAL = 0.05
    for req in workload:
        heapq.heappush(events, (req.id * ARRIVAL_INTERVAL, next(tiebreak), "arrive", req))

    arrival_of: dict[int, float] = {req.id: req.id * ARRIVAL_INTERVAL for req in workload}
    latencies: list[float] = []  # per-request latency, in finish order

    while events:
        now, _, kind, payload = heapq.heappop(events)
        if kind == "arrive":
            req: Request = payload
            b = pick_backend(backends, req)
            b.active += 1
            b.handled += 1
            finish_time = now + b.service_time(req)
            heapq.heappush(events, (finish_time, next(tiebreak), "finish", (b, req, now)))
        else:  # finish
            b, req, started_at = payload
            b.active -= 1
            latency = now - arrival_of[req.id]
            b.total_latency += latency
            latencies.append(latency)

    return backends, latencies


## 1️⃣ Round Robin (baseline)

Same as Notebook 1: cycle through the list. Notice it has **no idea** that
`big-1` is 4× stronger than `small-1` — it treats them identically.


In [ ]:
class RoundRobinPicker:
    def __init__(self):
        self.i = 0
    def __call__(self, backends, req):
        b = backends[self.i % len(backends)]
        self.i += 1
        return b

rr_backends, rr_lat = simulate(workload, RoundRobinPicker())
for b in rr_backends:
    avg = b.total_latency / b.handled if b.handled else 0
    print(f"{b.name:8s} handled={b.handled:5d}  avg_latency={avg:.2f}s")


## 2️⃣ Weighted Round Robin

If `big-1` is 4× as strong as `small-1`, give it 4× as many turns. We encode
that by building a list where each backend appears `capacity` times, then
cycle through it.

This is what real load balancers like NGINX and HAProxy do when you set a
`weight=N` on a backend.


In [ ]:
class WeightedRoundRobinPicker:
    def __init__(self, backends_template):
        # Expand the list: big-1 appears 4x, small-1 once, etc.
        self.slots = []
        for b in backends_template:
            self.slots.extend([b.name] * int(b.capacity))
        self.i = 0

    def __call__(self, backends, req):
        name = self.slots[self.i % len(self.slots)]
        self.i += 1
        # Look up the live backend object by name
        return next(b for b in backends if b.name == name)


wrr_backends, wrr_lat = simulate(workload, WeightedRoundRobinPicker(fresh_backends()))
for b in wrr_backends:
    avg = b.total_latency / b.handled if b.handled else 0
    print(f"{b.name:8s} handled={b.handled:5d}  avg_latency={avg:.2f}s")


## 3️⃣ Least Connections

> Send the next request to whichever backend currently has the **fewest
> in-flight requests**.

This adapts automatically: if `small-1` gets stuck on a huge request, its
`active` counter stays high and the LB stops sending it work.

It's usually the best default when requests have very different costs.


In [ ]:
def least_connections(backends, req):
    # Prefer backends with fewer in-flight requests; tie-break by capacity.
    return min(backends, key=lambda b: (b.active, -b.capacity))

lc_backends, lc_lat = simulate(workload, least_connections)
for b in lc_backends:
    avg = b.total_latency / b.handled if b.handled else 0
    print(f"{b.name:8s} handled={b.handled:5d}  avg_latency={avg:.2f}s")


## 4️⃣ Random

Pick any backend uniformly at random. Sounds dumb, but it actually works
*surprisingly* well at scale because the law of large numbers smooths things
out. And it needs **zero shared state** — every LB node can decide
independently, which matters if you have a fleet of load balancers.


In [ ]:
def random_pick(backends, req):
    return random.choice(backends)

random.seed(123)
rand_backends, rand_lat = simulate(workload, random_pick)
for b in rand_backends:
    avg = b.total_latency / b.handled if b.handled else 0
    print(f"{b.name:8s} handled={b.handled:5d}  avg_latency={avg:.2f}s")


## 5️⃣ Power of Two Choices (P2C)

> Pick **two** backends at random, then send the request to whichever one is
> less busy.

This sounds almost as silly as plain random, but mathematically it's a huge
upgrade. The maximum load on any backend drops from `O(log N / log log N)`
(plain random) to `O(log log N)` — a famous result by Mitzenmacher (2001).

P2C is the algorithm of choice in modern load balancers (NGINX's
`random two least_conn`, Envoy's `LEAST_REQUEST` with `choice_count=2`, HAProxy's
`first` with `random`). Why? It needs **almost no shared state** (each LB just
picks two backends and reads their counter), but behaves nearly as well as full
least-connections.

Because our fleet has **uneven capacity**, we score by `active / capacity` so
P2C doesn't blindly favor small backends just because they have fewer
connections.


In [ ]:
def power_of_two(backends, req):
    a, b = random.sample(backends, 2)
    # Score by load-per-capacity so unequal backends are compared fairly.
    score = lambda x: x.active / x.capacity
    return a if score(a) <= score(b) else b

random.seed(321)
p2c_backends, p2c_lat = simulate(workload, power_of_two)
for b in p2c_backends:
    avg = b.total_latency / b.handled if b.handled else 0
    print(f"{b.name:8s} handled={b.handled:5d}  avg_latency={avg:.2f}s")


## 🏁 Head-to-head comparison

Now that the simulator records the latency of **every individual request**, we
can compute the real metrics users care about:

- **avg** — the typical user experience.
- **p50** (median) — half of users are faster than this.
- **p95** — 95% of users are faster than this. This is the "tail". Tail
  latency is what makes a site feel slow even when the average looks fine,
  because each page view usually makes many requests and the slowest one
  dominates.
- **p99** — only 1% of users wait longer than this. Important for SLAs.


In [ ]:
import statistics

def percentiles(latencies: list[float]) -> dict[str, float]:
    """Return avg + key percentiles for a list of per-request latencies."""
    s = sorted(latencies)
    def pct(p):
        # Nearest-rank percentile — fine for a teaching notebook.
        k = max(0, min(len(s) - 1, int(round(p / 100.0 * len(s))) - 1))
        return s[k]
    return {
        "avg": statistics.mean(s),
        "p50": pct(50),
        "p95": pct(95),
        "p99": pct(99),
    }

results = {
    "round_robin":          percentiles(rr_lat),
    "weighted_round_robin": percentiles(wrr_lat),
    "least_connections":    percentiles(lc_lat),
    "random":               percentiles(rand_lat),
    "power_of_two":         percentiles(p2c_lat),
}

print(f"{'algorithm':22s} {'avg':>8s} {'p50':>8s} {'p95':>8s} {'p99':>8s}")
print("-" * 58)
for name, m in results.items():
    print(f"{name:22s} {m['avg']:8.2f} {m['p50']:8.2f} {m['p95']:8.2f} {m['p99']:8.2f}")


In [ ]:
import matplotlib.pyplot as plt

names = list(results.keys())
avgs  = [results[n]["avg"] for n in names]
p95s  = [results[n]["p95"] for n in names]

fig, ax = plt.subplots(figsize=(10, 4))
x = range(len(names))
ax.bar([i - 0.2 for i in x], avgs, width=0.4, label="average latency", color="#4C9AFF")
ax.bar([i + 0.2 for i in x], p95s, width=0.4, label="p95 latency",     color="#FF5630")
ax.set_xticks(list(x))
ax.set_xticklabels(names, rotation=15)
ax.set_ylabel("seconds")
ax.set_title("Algorithm comparison (lower is better)")
ax.legend()
plt.tight_layout()
plt.show()


## 🧠 Takeaways & when to use what

| Algorithm | Best when… | Watch out for… |
|---|---|---|
| **Round Robin** | backends are identical and requests are uniform | uneven request sizes → one backend gets "unlucky" |
| **Weighted Round Robin** | backends have known, different capacities | capacities change over time (autoscaling) |
| **Least Connections** | requests have very different costs (APIs, DB queries) | requires the LB to track in-flight state per backend |
| **Random** | very large fleets, or stateless LB nodes | small fleets → high variance |
| **Power of Two Choices (P2C)** | you want least-connections quality with almost no coordination | needs at least 2 backends; weight by capacity if fleet is uneven |

> 💡 In real life, most modern L7 load balancers (NGINX, Envoy, HAProxy, AWS
> ALB) default to **least connections** or **power-of-two-choices**. P2C is
> the secret sauce behind a lot of "magic" load balancers — it's almost as good
> as least-connections but works without any coordination between LB instances.

👉 Next: in **Notebook 3** we'll see what happens when a backend gets sick,
how **health checks** remove it from the pool, and why **sticky sessions**
are a double-edged sword. Then **Notebook 4** tackles **consistent hashing**
— how to do sticky routing without losing everyone's session every time you
add or remove a backend.
